<a href="https://colab.research.google.com/github/vitor-py/fraud-detection-random-forest/blob/main/Detec%C3%A7%C3%A3oFraude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MODELO RANDOM FOREST PARA DETECÇÃO DE FRAUDE**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [39]:
data = pd.read_csv('/home/card_transdata.csv')
data.head(5)

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


De primeira consulta nos dados, optei por ver rápidamente se havia dados nulos que pudessem prejudicar o treinamento do modelo.

In [ ]:
data.isnull().sum()

Como havia dados nulos, fiz a limpeza de imediato!

In [ ]:
data = data.dropna()

In [ ]:
data.isnull().sum()

Agora usarei um pacote para análise explorátoria de dados automático, ele fará um report completo.

 (output retirado para não 'bugar' o github)

In [ ]:
!pip install -q ydata-profiling
from ydata_profiling import ProfileReport

profile = ProfileReport(data, explorative=True)
profile.to_notebook_iframe()

Partindo para a criação das features:

In [ ]:
from sklearn.model_selection import train_test_split
x = np.array(data[['distance_from_home',	'distance_from_last_transaction', 'ratio_to_median_purchase_price','repeat_retailer',	'used_chip',	'used_pin_number',	'online_order']])
y = np.array(data['fraud'])

# **Treinamento do modelo!**
**detalhes:**

Por ser um dataset desbalanceado, optei por estratificar o target para que a proporção de classes seja mantida igual tanto no conjunto de treino quanto no de teste.

--------------------------------------------
Utilizei 200 decisions tree \
O nível de profundidade limitei a 12 para evitar overfiting por parte de árvores mais longas \
Ainda me preocupando com o desbalanceamente, utilizei o class_weight='balanced' \
E por fim, a seed escolhida é simplesmente a *resposta para tudo*





In [ ]:
from sklearn.ensemble import RandomForestClassifier
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, stratify=y)
model = RandomForestClassifier(n_estimators=200, max_depth=12,class_weight='balanced',random_state=42)
model.fit(xtrain, ytrain)

In [ ]:
y_pred = model.predict(xtest)

# **RESULTADOS**

Tudo está perfeito! (até demais)

a precisão deste modelo beira os 100% \
isso pode significar que alguma variável em específico está tendo um peso muito grande para a detecção da fraude.


In [34]:
print('Métricas do Classification Report: \n', classification_report(ytest, y_pred))
print('Confusion Matrix: \n', confusion_matrix(ytest, y_pred))

Métricas do Classification Report: 
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      7527
         1.0       1.00      1.00      1.00       722

    accuracy                           1.00      8249
   macro avg       1.00      1.00      1.00      8249
weighted avg       1.00      1.00      1.00      8249

Confusion Matrix: 
 [[7527    0]
 [   1  721]]


Tendo isso em mente, decido por ver a importância de cada feature em meu modelo, para enfim descobrir quem está causando isso.

-----------------------------------------------
Abaixo, no resultado do código é possível ver que a feature 'ratio_to_median_purchase_price' está sendo a variável responsável. \
Muito provávelmente essa artificialidade ocorre por ser um dataset vindo do Kaggle **(créditos ao final do notebook)**.

In [35]:
features = [
'distance_from_home',
'distance_from_last_transaction',
'ratio_to_median_purchase_price',
'repeat_retailer',
'used_chip',
'used_pin_number',
'online_order'
]

importances = model.feature_importances_

df = pd.DataFrame({
'feature': features,
'importance': importances
}).sort_values('importance', ascending=False)

print(df)

                          feature  importance
2  ratio_to_median_purchase_price    0.538950
0              distance_from_home    0.205494
6                    online_order    0.106400
1  distance_from_last_transaction    0.083994
5                 used_pin_number    0.029985
4                       used_chip    0.027260
3                 repeat_retailer    0.007918


# **TESTES COM DADOS NOVOS**


Neste primeiro teste eu utilizei dados que apontam uma fraude óbvia (me baseando na importância de cada feature) \
**resultado:** Modelo descobriu a fraude com sucesso

In [ ]:
teste = np.array([[95.432187,	38.774512,	7.893421,	0.0,	0.0,	0.0,	1.0]])

In [36]:
model.predict(teste)

array([1.])

Neste segundo teste eu utilizei dados que apontam uma NÃO fraude óbvia (me baseando na importância de cada feature) \
**resultado:** Modelo não apontou como fraude, sucesso.

In [ ]:
teste2 = np.array([[3.482951,	0.842113,	0.967421,	1.0,	1.0,	1.0,	0.0]])

In [37]:
model.predict(teste2)

array([0.])

# **ADVERSARIAL EXAMPLE**

hora de observar se o modelo consegue prever fraudes não óbvias, mais desafiadoras. \
 Aqui neste array abaixo se encontra uma **fraude**, todavia não apliquei tanto volume nas features de peso. \
**resultado:** modelo não descobriu fraude

In [ ]:
teste3 = np.array([[4.213584,	1.129774,	1.082341,	1.0,	1.0,	0.0,	0.0]])

In [38]:
model.predict(teste3)

array([0.])

# **CONCLUSÃO:**
No geral, o modelo apresentou um desempenho muito satisfatório dentro das condições do dataset utilizado. A Random Forest demonstrou alta capacidade de identificar padrões associados a transações fraudulentas, alcançando métricas de precisão e recall bastante elevadas no conjunto de teste.

Observou-se que algumas variáveis tiveram maior impacto nas decisões do modelo, principalmente ratio_to_median_purchase_price, distance_from_home e online_order, indicando que transações com valores muito acima do padrão do cliente, realizadas a grandes distâncias de sua localização habitual ou realizadas online tendem a apresentar maior probabilidade de fraude.

Durante os testes também foi possível verificar que o modelo consegue detectar com facilidade fraudes que seguem os padrões presentes no dataset. Entretanto, quando são criadas transações fraudulentas que se comportam de maneira muito semelhante a transações legítimas, por exemplo, compras próximas da residência do usuário, em estabelecimentos recorrentes e com valores próximos da média, o modelo tende a classificá-las como não fraude.

Portanto, para aplicações reais, seria interessante complementar o modelo com mais variáveis comportamentais, maior diversidade de dados e técnicas adicionais de engenharia de atributos. Dessa forma, seria possível capturar padrões de fraude mais complexos e melhorar ainda mais a robustez do sistema de detecção.

Em resumo, o modelo mostrou-se eficiente para identificar fraudes dentro dos padrões presentes no dataset, além de demonstrar como técnicas de aprendizado de máquina podem ser aplicadas de forma prática na detecção de atividades suspeitas em transações financeiras.

DATASET: https://www.kaggle.com/datasets/dhanushnarayananr/credit-card-fraud